In [1]:
import os
import re
import torch 
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import torch.optim as optim
from scipy.signal import welch
from scipy.stats import entropy
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Analyzing dataset
dataset = "./Dataset"
dataset_actions = {}

for root, dirs, files in os.walk(dataset):
    for file in files:
        file = file[:-4]
        action =  re.sub(r'\d+$', '', file)
        if action not in dataset_actions:
            dataset_actions[action] = 1
        else:
            dataset_actions[action] += 1

Dataset_actions_log = "Dataset Actions Count\n"
total = 0
for key, value in dataset_actions.items():
    total += value
    Dataset_actions_log += f"{key}: {value} | "
Dataset_actions_log += f"total: {total}"
print(Dataset_actions_log)

Dataset Actions Count
final: 175 | grenade: 195 | idle: 21 | reload: 172 | shield: 195 | total: 758


In [3]:
# Read in dataset into a dataframe of 7 columns, 6 measurement columns, 1 label
# each row has 6 lists, 1 label corresponding to 1 file
dataset = "./Dataset"
dataset_pd = pd.DataFrame(columns=["x_acc", "y_acc", "z_acc", "x_gyro", "y_gyro", "z_gyro", "action"])

for root, dirs, files in os.walk(dataset):
    for file in files:
        if 'idle' not in file:
            file_path = os.path.join(root, file)
            file_pd = pd.read_csv(file_path)
            x_acc = file_pd.iloc[:, 0].tolist()
            y_acc = file_pd.iloc[:, 1].tolist()
            z_acc = file_pd.iloc[:, 2].tolist()
            x_gyro = file_pd.iloc[:, 3].tolist()
            y_gyro = file_pd.iloc[:, 4].tolist()
            z_gyro = file_pd.iloc[:, 5].tolist()
            file = file[:-4]
            action =  re.sub(r'\d+$', '', file)
            row = {"x_acc": x_acc, "y_acc": y_acc, "z_acc": z_acc, "x_gyro": x_gyro, "y_gyro": y_gyro, "z_gyro": z_gyro, "action": action}
            dataset_pd = pd.concat([dataset_pd, pd.DataFrame([row])], ignore_index=True)
        else: 
            for chunk_idle in pd.read_csv(os.path.join(root, file), chunksize=200): # take idle at 200 row max length
                x_acc = chunk_idle.iloc[:, 0].tolist()
                y_acc = chunk_idle.iloc[:, 1].tolist()
                z_acc = chunk_idle.iloc[:, 2].tolist()
                x_gyro = chunk_idle.iloc[:, 3].tolist()
                y_gyro = chunk_idle.iloc[:, 4].tolist()
                z_gyro = chunk_idle.iloc[:, 5].tolist()
                file = file[:-4]
                action =  re.sub(r'\d+$', '', file)
                row = {"x_acc": x_acc, "y_acc": y_acc, "z_acc": z_acc, "x_gyro": x_gyro, "y_gyro": y_gyro, "z_gyro": z_gyro, "action": 'idle'}
                dataset_pd = pd.concat([dataset_pd, pd.DataFrame([row])], ignore_index=True)


In [4]:
print(dataset_pd.shape)

(855, 7)


In [5]:
# Extract all features (can be removed later)
# time domain: mean, std, rms, min, max, median, 25th percentile, 50th percentile, 75th percentile, 
# zero-crossing rate, peak count
# frequency domain: mean, max, Spectral Entropy, total power, Spectral centroid, DC component, 
# Dominant frequency, Power Spectral Density, average energy per coefficient 

all_features = pd.DataFrame()
for i in range(len(dataset_pd.index)):
    data_dict = {}
    label = dataset_pd.iloc[i, len(dataset_pd.columns)-1]
    for j in range(len(dataset_pd.columns)-1):
        col_name = dataset_pd.columns[j]
        data = dataset_pd.iloc[i, j]
        # Compute time-domain features
        t_mean = np.mean(data)
        t_std_deviation = np.std(data)
        t_rms = np.sqrt(np.mean(np.square(data)))
        t_min = np.min(data)
        t_max = np.max(data)
        t_median = np.median(data)
        t_25_q = np.percentile(data, 25)
        t_50_q = np.percentile(data, 50)
        t_75_q = np.percentile(data, 75)
        t_zero_crossing = np.sum(np.diff(np.sign(data)) != 0)
        t_peaks = len(np.where((np.diff(np.sign(np.diff(data)))) < 0)[0])

        # Compute frequency-domain features
        freq_domain = np.fft.rfft(data)
        f_mean = abs(np.mean(freq_domain))
        f_max = abs(max(freq_domain))
        f_spectral_entropy = entropy(np.abs(freq_domain))
        f_total_power = np.sum(np.abs(freq_domain) ** 2)
        f_spectral_centroid = np.sum(np.arange(len(freq_domain)) * np.abs(freq_domain)) / np.sum(np.abs(freq_domain))
        f_DC_component = abs(freq_domain[0])
        f_dominant_frequency = np.argmax(np.abs(freq_domain))
        f_PSD = welch(data)[1]
        f_average_E_per_coeff = f_total_power / len(freq_domain)

        # Create dictionary
        data_dict.update({
            f"t_mean_{col_name}": t_mean,
            f"t_std_deviation_{col_name}": t_std_deviation,
            f"t_rms_{col_name}": t_rms,
            #f"t_min_{col_name}": t_min,
            #f"t_max_{col_name}": t_max,
            #f"t_median_{col_name}": t_median,
            #f"t_25_q_{col_name}": t_25_q,
            #f"t_50_q_{col_name}": t_50_q,
            f"t_75_q_{col_name}": t_75_q,
            #f"t_zero_crossing_{col_name}": t_zero_crossing,
            #f"t_peaks_{col_name}": t_peaks,
            #"freq_domain": freq_domain,
            f"f_mean_{col_name}": f_mean,
            f"f_max_{col_name}": f_max,
            #f"f_spectral_entropy_{col_name}": f_spectral_entropy,
            #f"f_total_power_{col_name}": f_total_power,
            f"f_spectral_centroid_{col_name}": f_spectral_centroid,
            #"f_DC_component": f_DC_component,
            #f"f_dominant_frequency_{col_name}": f_dominant_frequency,
            #"f_PSD": f_PSD,
            f"f_average_E_per_coeff_{col_name}": f_average_E_per_coeff,
        })
    data_dict.update({f"label": label})
    
    all_features = pd.concat([all_features, pd.DataFrame([data_dict])], ignore_index=True)

label_encoder = LabelEncoder()
all_features['label'] = label_encoder.fit_transform(all_features['label'])
for label, encoded_label in zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)):
    print(f"{label}: {encoded_label}")

print(all_features.shape)

final: 0
grenade: 1
idle: 2
reload: 3
shield: 4
(855, 49)


In [6]:
class Sample_NN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(48, 24)  # Input layer: 6 neurons, Hidden Layer: 4 neurons
        self.fc2 = nn.Linear(24, 12)  # Hidden layer: 4 neurons, Output Layer: 2 neurons
        self.fc3 = nn.Linear(12, 5)


    def forward(self, x):
        x = self.fc1(x)
        x = F.leaky_relu(x)
        x = self.fc2(x)
        x = F.leaky_relu(x)
        x = self.fc3(x)
        output = F.softmax(x, dim=1)  # dimension 1: softmax function applied along second dimension of the tensor
        return output

In [7]:
class Sample_NN_dataset(Dataset):
    def __init__(self, all_features):
        # preprocessing done here
        self.data = all_features
        self.labels = self.data['label']
        self.data = self.data.drop(columns=['label'])
        # L2 Normalisation for each column
        self.data = self.data / np.linalg.norm(self.data, axis=0)
        # Scale to same range
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(self.data)
        min_max_scaler = MinMaxScaler(feature_range=(0, 1))
        scaled_min_max_data = min_max_scaler.fit_transform(scaled_data)
        self.data = pd.DataFrame(scaled_min_max_data, columns=self.data.columns)


    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        features = torch.tensor(self.data.iloc[idx].values, dtype=torch.float)
        label = torch.tensor(self.labels[idx], dtype=torch.long)  # Assuming labels are integers
        return features, label


In [10]:
dataset = Sample_NN_dataset(all_features)
train_data, test_data = train_test_split(dataset, test_size=0.25, random_state=1334)  # my favourite number
batch_size = 32
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
epochs = 100
model = Sample_NN()
optimizer = optim.SGD(model.parameters(), lr=0.1)
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
    model.train()  # Sets the model to training mode
    for inputs, labels in train_loader:
        optimizer.zero_grad()  # Zero the gradients
        outputs = model(inputs)  # Forward pass
        loss = criterion(outputs, labels)  # Calculate the loss
        loss.backward()  # Backward pass
        optimizer.step()  # Update weights

    model.eval()  # Sets the model to evaluation mode
    with torch.no_grad():  # Disable gradient calculation during testing
        correct = 0
        total = 0
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)  # we dont need all the maximum of a tensor, hence _
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total

    print(f'Epoch [{epoch + 1}/{epochs}], Test Accuracy: {accuracy:.2%}')

Epoch [1/100], Test Accuracy: 20.56%
Epoch [2/100], Test Accuracy: 20.56%
Epoch [3/100], Test Accuracy: 20.56%
Epoch [4/100], Test Accuracy: 20.56%
Epoch [5/100], Test Accuracy: 20.56%
Epoch [6/100], Test Accuracy: 40.19%
Epoch [7/100], Test Accuracy: 33.64%
Epoch [8/100], Test Accuracy: 39.25%
Epoch [9/100], Test Accuracy: 32.71%
Epoch [10/100], Test Accuracy: 29.44%
Epoch [11/100], Test Accuracy: 24.30%
Epoch [12/100], Test Accuracy: 27.57%
Epoch [13/100], Test Accuracy: 22.43%
Epoch [14/100], Test Accuracy: 24.30%
Epoch [15/100], Test Accuracy: 25.23%
Epoch [16/100], Test Accuracy: 22.43%
Epoch [17/100], Test Accuracy: 22.43%
Epoch [18/100], Test Accuracy: 22.43%
Epoch [19/100], Test Accuracy: 22.43%
Epoch [20/100], Test Accuracy: 22.43%
Epoch [21/100], Test Accuracy: 22.43%
Epoch [22/100], Test Accuracy: 22.90%
Epoch [23/100], Test Accuracy: 22.90%
Epoch [24/100], Test Accuracy: 22.43%
Epoch [25/100], Test Accuracy: 29.91%
Epoch [26/100], Test Accuracy: 35.51%
Epoch [27/100], Test 

In [17]:
torch.save(model.state_dict(), 'model_weights_biases.pth')

In [18]:
model_weights_path = 'model_weights_biases.pth'
model.load_state_dict(torch.load(model_weights_path))

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.data)

fc1.weight tensor([[ 0.1200,  0.0645, -0.0867,  ...,  0.0662, -0.1146, -0.0337],
        [ 0.0695, -0.2063,  0.2153,  ...,  0.0086,  0.0694, -0.2245],
        [ 0.3626,  0.0231, -0.2577,  ..., -0.0194,  0.0884, -0.1103],
        ...,
        [-0.1284,  0.1194,  0.1374,  ...,  0.1128, -0.0171,  0.0154],
        [-0.1044,  0.0438, -0.0968,  ..., -0.0470,  0.1292, -0.0900],
        [ 0.0263, -0.1749,  0.1931,  ..., -0.0894, -0.0029, -0.1016]])
fc1.bias tensor([-0.1214,  0.4248, -0.2026,  0.0417,  0.0321, -0.0413,  0.2079,  0.0690,
         0.1122,  0.2230, -0.0684,  0.3228,  0.2519, -0.0983, -0.0244,  0.5139,
        -0.4811,  0.5352, -0.0478, -0.1331,  0.0200, -0.0198,  0.0466,  0.2689])
fc2.weight tensor([[ 7.1071e-03,  2.2958e-01, -9.3431e-01,  1.6340e-01, -6.4309e-02,
         -1.5321e-01,  8.1601e-02,  1.4766e-01, -2.2185e-02,  8.1129e-01,
          1.4098e-01,  4.7326e-01, -7.0638e-02, -8.9680e-02, -3.8071e-01,
         -2.8364e-01,  6.0482e-02,  3.7331e-01, -1.7629e-02,  1.1701e-01